# 1. SVC

In [ ]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.datasets import make_moons  
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.linear_model import LogisticRegression 
from sklearn.metrics import accuracy_score, classification_report 


In [ ]:
# Generate moon dataset
X, y = make_moons(n_samples=200, random_state=0, shuffle=True)

# Split the dataset into training and testing sets
X_train, X_test = X[:100], X[100:]
y_train, y_test = y[:100], y[100:]


In [ ]:
class CustomSVC:
    def __init__(self, kernel='rbf', C=1.0, gamma='scale', degree=3):
        self.kernel = kernel
        self.C = C
        self.gamma = gamma
        self.degree = degree
        self.model = None

    def fit(self, X, y):
        self.model = SVC(kernel=self.kernel, C=self.C, gamma=self.gamma, degree=self.degree)
        self.model.fit(X, y)

    def predict(self, X):
        return self.model.predict(X)



In [ ]:
# Create an instance of CustomSVC with different kernel options
kernels = ['linear', 'poly', 'rbf', 'sigmoid' ] 

# Plotting decision boundary
def plot_decision_boundary(model, X, y, kernel):
    h = 0.01
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    plt.contourf(xx, yy, Z, alpha=0.8)
    plt.scatter(X[:, 0], X[:, 1], c=y, s=50, edgecolors='black')
    plt.scatter(
    model.model.support_vectors_[:, 0], 
    model.model.support_vectors_[:, 1], 
    s=50, facecolors='none', edgecolors='red' )
    plt.xlabel('X1')
    plt.ylabel('X2')
    plt.title(kernel)
    plt.show()

# Plot decision boundaries for each kernel
for kernel in kernels:
    model = CustomSVC(kernel=kernel)
    model.fit(X_train, y_train)
    plot_decision_boundary(model, X_test, y_test, kernel)


The linear kernel assumes a linear decision boundary. It works well when the data is linearly separable and the decision boundary will be a straight line. On the other hand the polynomial kernel allows for more flexible decision boundaries by using higher-degree polynomials. A higher degree can capture more complex patterns in the data but may also lead to overfitting. <br>

The sigmoid kernel is based on the sigmoid function and can handle non-linear decision boundaries. due to its sensitivity to the choice of hyperparameters it is less commonly used compared to the other kernels.<br>

The moon dataset is not linearly separable; it requires a non-linear decision boundary to accurately separate the two moon shapes. The 'rbf' kernel is well-suited for capturing non-linear relationships in the data. It can create complex decision boundaries by mapping the data points to a higher-dimensional space using a Gaussian function. Therefore in this case rbf perfomed the best. <br>

# 2. Model Evaluation

In [ ]:
# Load the dataset
data = pd.read_csv('breast-cancer.csv')

data.head() 

In [ ]:
data.shape 

In [ ]:
data.info()

In [ ]:
# Separate features and target variable
X = data.drop(['id', 'diagnosis'], axis=1)
y = data['diagnosis']

# Perform feature scaling using StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Check the skewness of the features 
Xscaled = pd.DataFrame(X_scaled)
Xscaled.skew() 


the positive skewness values indicate a right-skewed distribution. it means the majority of the data is concentrated on the left side of the distribution, and the tail extends towards the right.  


In [ ]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=0) 

In [ ]:
# Model 1: Logistic Regression with default hyperparameters
lr_model_1 = LogisticRegression()
lr_model_1.fit(X_train, y_train)

# Model 2: Logistic Regression with different C value
lr_model_2 = LogisticRegression(C=0.1)
lr_model_2.fit(X_train, y_train)

# Model 3: Logistic Regression with different regularization penalty (L1)
lr_model_3 = LogisticRegression(penalty='l1', solver='liblinear')
lr_model_3.fit(X_train, y_train)


In [ ]:
# Model 4: SVM with default hyperparameters
svm_model_1 = SVC()
svm_model_1.fit(X_train, y_train)

# Model 5: SVM with different C value and linear kernel
svm_model_2 = SVC(C=0.1, kernel='linear')
svm_model_2.fit(X_train, y_train)

# Model 6: SVM with different gamma value and RBF kernel
svm_model_3 = SVC(gamma=0.1, kernel='rbf')
svm_model_3.fit(X_train, y_train)


In [ ]:
# Create a list of models
models = [lr_model_1, lr_model_2, lr_model_3, svm_model_1, svm_model_2, svm_model_3]

# Evaluate each model
for model in models:
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred) 
    print("Model:", model)
    print("Accuracy:", accuracy)
    print("Classification Report:")
    print(report)
    print("------------------------------")


Accuracy: It measures the overall correctness of the model's predictions. Higher accuracy indicates that the model is making correct predictions more often.

Precision: It is the ratio of true positives to the sum of true positives and false positives. In the context of breast cancer diagnosis, precision represents the ability of the model to correctly identify malignant cases (M). Higher precision indicates fewer false positives, which means the model is correctly identifying actual positive cases.

Recall: It is the ratio of true positives to the sum of true positives and false negatives. In the context of breast cancer diagnosis, recall represents the ability of the model to correctly identify all actual positive cases. Higher recall indicates fewer false negatives, which means the model is correctly capturing most positive cases.

F1-score: It provides a balanced measure that considers both precision and recall. Higher F1-score indicates a better balance between precision and recall.

 In the case of breast cancer diagnosis, the focus is often on minimizing false negatives (missing actual positive cases). Therefore, we would prioritize models with higher recall values while maintaining a reasonable level of precision. Among the evaluated models, SVC with default parameters (Model: SVC()) achieved the highest recall for the (M) class while maintaining a high overall accuracy.

This assignment has been done with help of Samaneh Shahpouri